# 11 — FICHE D'IDENTITÉ par rythme

Une visualisation unifiée qui rassemble toute l'analyse d'un rythme sur une page :
- **rythme de base** : les motifs du dunun et de l'accompagnement, juxtaposés sur le cycle ;
- **swing** : la non-isochronie des subdivisions (court/moyen/long) ;
- **intensité musicale** : une métrique 0–100 (densité de frappes + énergie + tempo) au fil du morceau ;
- **grammaire du soliste** : quel type de frappe (bass/tone/slap) à chaque position.

Tout vient de `djembe_utils` (fonctions `subdivision_profile`, `swing_ratio`, `intensity_curve`).

## 0. Imports

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import djembe_utils as dj
c = dj.list_corpus()
score = pd.read_csv('output/partitions.csv')   # produit par 06_partitions
COLP = dict(zip(dj.PIECES, ['#4a8cff', '#e05ce0', '#3ddb7a']))

def prof_mean(piece, code, takes):
    ps = [p.values for t in takes if (p := dj.subdivision_profile(piece, int(t), code)) is not None]
    return np.mean(ps, axis=0) if ps else np.zeros(12)

## 1. La fiche d'un rythme
Change `PIECE`. Quatre panneaux = quatre facettes du rythme.

In [ ]:
PIECE = 'Woloso'   # <- 'Manjanin', 'Maraka', 'Woloso'
takes = c.loc[c['piece']==PIECE, 'take'].unique()
col = COLP[PIECE]
fig, ax = plt.subplots(2, 2, figsize=(15, 9)); fig.suptitle(f'FICHE — {PIECE}', fontsize=16, weight='bold')

# (a) rythme de base : dunun + accompagnement juxtaposés
ax[0,0].step(range(1,13), prof_mean(PIECE,'D1',takes), where='mid', color='#2b6cff', lw=2.5, label='dunun')
ax[0,0].step(range(1,13), prof_mean(PIECE,'J2',takes), where='mid', color='#9b59b6', lw=2.5, label='jembé accomp')
ax[0,0].set(title='Rythme de base (juxtaposition)', xlabel='subdivision', ylim=(-.05,1.2)); ax[0,0].set_xticks(range(1,13)); ax[0,0].legend()

# (b) swing
sw = dj.swing_ratio(PIECE, takes)
ax[0,1].bar(['1re','2e','3e'], sw, color=col); ax[0,1].axhline(100/3, ls='--', color='k', alpha=.5)
ax[0,1].set(title=f'Swing : {sw[0]:.0f} / {sw[1]:.0f} / {sw[2]:.0f}  (isochrone=33/33/33)', ylabel='% de la pulsation')

# (c) intensité musicale (1re prise)
ic = dj.intensity_curve(PIECE, int(takes[0]))
ax[1,0].plot(ic['t'], ic['intensity'], color=col, lw=2); ax[1,0].fill_between(ic['t'], ic['intensity'], alpha=.2, color=col)
ax[1,0].set(title=f'Intensité musicale : {ic["intensity"].iloc[:10].mean():.0f} → {ic["intensity"].iloc[-10:].mean():.0f} / 100',
            xlabel='temps (s)', ylim=(0,105))

# (d) grammaire du soliste
sub = score[score['piece']==PIECE]; tab = pd.crosstab(sub['SD'], sub['J1'], normalize='index')
SC = {'b':'#2b6cff','t':'#d63bd6','s':'#2ec45f'}; bottom = np.zeros(12)
for s in ['b','t','s']:
    if s in tab: ax[1,1].bar(tab.index, tab[s], bottom=bottom, color=SC[s], label=s); bottom += tab[s].values
ax[1,1].set(title='Grammaire du soliste (frappe par position)', xlabel='subdivision'); ax[1,1].set_xticks(range(1,13)); ax[1,1].legend(title='frappe')
plt.tight_layout(); plt.savefig(f'output/11_fiche_{PIECE}.png', dpi=130); plt.show()

## 2. Comparer les 3 rythmes d'un coup d'œil

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(16, 11))
for i, piece in enumerate(dj.PIECES):
    tk = c.loc[c['piece']==piece, 'take'].unique(); col = COLP[piece]
    a0, a1, a2 = axes[i]
    a0.step(range(1,13), prof_mean(piece,'D1',tk), where='mid', color='#2b6cff', lw=2, label='dunun')
    a0.step(range(1,13), prof_mean(piece,'J2',tk), where='mid', color='#9b59b6', lw=2, label='accomp')
    a0.set(title=f'{piece} — rythme de base', ylim=(-.05,1.2)); a0.set_xticks(range(1,13)); a0.legend(fontsize=8)
    sw = dj.swing_ratio(piece, tk)
    a1.bar(['1re','2e','3e'], sw, color=col); a1.axhline(100/3, ls='--', color='k', alpha=.5)
    a1.set(title=f'swing {sw[0]:.0f}/{sw[1]:.0f}/{sw[2]:.0f}', ylabel='%')
    ic = dj.intensity_curve(piece, int(tk[0]))
    a2.plot(ic['t'], ic['intensity'], color=col, lw=2); a2.fill_between(ic['t'], ic['intensity'], alpha=.2, color=col)
    a2.set(title=f'intensité {ic["intensity"].iloc[:10].mean():.0f}→{ic["intensity"].iloc[-10:].mean():.0f}', xlabel='temps (s)', ylim=(0,105))
plt.tight_layout(); plt.savefig('output/11_fiches_rythmes.png', dpi=130); plt.show()

## Bilan
- Une **fiche unique** par rythme : motif de base + swing + intensité + grammaire du soliste.
- L'**intensité monte** systématiquement (~20 → ~90) : la musique de jembé est une montée en énergie.
- Le **swing** distingue Maraka (36/28/36) de Manjanin/Woloso (~25/34/41).
- C'est l'analyse complète « par rythme » demandée. Reste `12_decomposeur` : faire tout ça automatiquement sur un audio quelconque.